# Write and Round-Trip

`CSV.write` saves any Tables.jl source as delimited text. This tutorial writes a named tuple of vectors with different options, appends to a file, compresses it and checks which values survive a write and a read.

It needs CSV.jl and Tables.jl.

In [ ]:
using CSV, Tables, Dates

cd(mktempdir())
data = (
	id    = [1, 2, 3],
	price = [9.5, missing, 12.0],
	city  = ["Lisbon", "Porto", "Faro"],
	date  = Date(2024, 1, 5) .+ Day.(0:2),
)

## Write with Defaults

Commas between fields, a header row, missing values as empty fields and ISO dates.

In [ ]:
CSV.write("out.csv", data)
print(read("out.csv", String))

## Options

A tab-separated file with `NA` for missing values and day-first dates.

In [ ]:
CSV.write("out.tsv", data; delim = '\t', missingstring = "NA", dateformat = "dd/mm/yyyy")
print(read("out.tsv", String))

## Append Rows

`append = true` adds rows to an existing file without writing the header again, useful for logs and batch results.

In [ ]:
more = (id = [4], price = [7.25], city = ["Braga"], date = [Date(2024, 1, 8)])
CSV.write("out.csv", more; append = true)
print(read("out.csv", String))

## Compress

`compress = true` writes gzip. CSV.jl reads `.gz` files directly.

In [ ]:
big = (x = 1:100_000, y = repeat(["a", "b", "c", "d"], 25_000))
CSV.write("big.csv", big)
CSV.write("big.csv.gz", big; compress = true)
println("Plain: ", filesize("big.csv"), " bytes, gzip: ", filesize("big.csv.gz"), " bytes")
println("Rows read back from gzip: ", length(CSV.File("big.csv.gz")))

## A Matrix

A matrix has no column names, so wrap it with `Tables.table` and give them.

In [ ]:
M = [1.0 2.0; 3.0 4.0; 5.0 6.0]
CSV.write("matrix.csv", Tables.table(M; header = [:a, :b]))
print(read("matrix.csv", String))

## Round Trip

CSV stores text only. Write a table, read it back and compare every column: codes with leading zeros stored as numbers, or text that looks like a number, do not come back the same.

In [ ]:
tricky = (
	code  = ["00123", "04567", "08910"],
	flag  = ["yes", "no", "yes"],
	value = [0.1 + 0.2, 1 / 3, 2.5],
)
back = Tables.columntable(CSV.File(IOBuffer(sprint(io -> CSV.write(io, tricky)))))
for name in keys(tricky)
	println(rpad(name, 8), isequal(tricky[name], back[name]) ? "identical" : "changed", "  ", eltype(tricky[name]), " -> ", eltype(back[name]))
end

The `code` column came back as integers without the zeros. Force the type when reading it back:

In [ ]:
back = Tables.columntable(CSV.File(IOBuffer(sprint(io -> CSV.write(io, tricky))); types = Dict(:code => String)))
println(back.code == tricky.code)

## Next Steps

- The panel's **Round-Trip Check** runs this comparison on your own table.
- **Save as Excel Workbook** writes the same table to .xlsx for spreadsheet users.